### Базовые операции со столбцами DataFrame ###

In [ ]:
import numpy as np
import pandas as pd
melb_data = pd.read_csv('data/melb_data_ps.csv', sep=',')

melb_data.head()


In [ ]:
melb_df = melb_data.copy()
melb_df.head()

melb_df = melb_df.drop(['index', 'Coordinates'], axis = 1)
melb_df.head()

# melb_df.drop(['index','Coordinates'], axis=1, inplace=True)
# melb_df.head()

### Операции со столбцами ###

In [ ]:
total_rooms = melb_df['Rooms'] + melb_df['Bedroom'] + melb_df['Bathroom']
display(total_rooms)

In [ ]:
melb_df['Mean_Room_Area'] = melb_df['BuildingArea'] / total_rooms
display(melb_df['Mean_Room_Area'])

In [ ]:
diff_area = melb_df['BuildingArea'] - melb_df['Landsize']
sum_area = melb_df['BuildingArea'] + melb_df['Landsize']
melb_df['AreaRatio'] = diff_area/sum_area
display(melb_df['AreaRatio'])

In [ ]:
countries_df = pd.DataFrame({
    'country': ['Англия', 'Канада', 'США', 'Россия', 'Украина', 'Беларусь', 'Казахстан'],
    'population': [56.29, 38.05, 322.28, 146.24, 45.5, 9.5, 17.04],
    'area': [133396, 9984670, 9826630, 17125191, 603628, 207600, 2724902]
})

"""Для каждой страны рассчитайте плотность населения (количество человек на квадратный километр).
Затем по полученным данным рассчитайте среднее по плотностям населения в указанных странах. Ответ округлите до сотых."""

countries_df['area_mln'] = countries_df['area'] / 1_000_000
countries_df = countries_df.drop(['area'], axis = 1)
display(countries_df)

countries_df['density'] = countries_df['population'] / countries_df['area_mln']
display(countries_df)

display(round(countries_df['density'].mean(), 2))

In [ ]:
display(melb_df['Date'])

melb_df['Date'] = pd.to_datetime(melb_df['Date'], dayfirst=True)
display(melb_df['Date'])




### Дата и время ###

In [ ]:
years_sold = melb_df['Date'].dt.year

print(years_sold)

print('Min year sold:', years_sold.min())
print('Max year sold:', years_sold.max())
print('Mode year sold:', years_sold.mode()[0])

melb_df['Month_Sale'] = melb_df['Date'].dt.month
print(years_sold)

melb_df['Month_Sale'].value_counts(normalize = True)

In [ ]:
display(melb_df['Date'])
melb_df['Date'] = pd.to_datetime(melb_df['Date'], dayfirst=True)
display(melb_df['Date'])

### Разница между временными интервалами ###

In [ ]:
delta_days = melb_df['Date'] - pd.to_datetime('2016-12-01') 
display(delta_days)


delta_days.dt.days

display(delta_days.dt.days)

# melb_df['AgeBuilding'] = melb_df['Date'].dt.year - melb_df['YearBuilt']
display(melb_df['AgeBuilding'])

# melb_df.drop(['YearBuilt'], axis = 1, inplace=True)

In [ ]:
"""Важно разобрать по деталям"""


melb_df['WeekdaySale'] = melb_df['Date'].dt.dayofweek
display(melb_df['WeekdaySale'])

def get_weekend(weekday):
    if weekday == 5 or weekday == 6:
        return 1
    else:
        return 0
    
melb_df['Weekend'] = melb_df['WeekdaySale'].apply(get_weekend)

mean_weekend_price = round(melb_df[melb_df['Weekend'] == 1]['Price'].mean(), 0)

print(mean_weekend_price)

# display(melb_df['WeekdaySale'].isin([5,6]).mode()[0])


# melb_df['WeekdaySale'].value_counts

# weekend_count  = melb_df['WeekdaySale'].isin([5,6]).mode()[0]

In [ ]:
"""Мы предлагаем вам самостоятельно разобраться с этой ошибкой: попробуйте написать функцию-преобразование (lambda-функцию-преобразование), 
которая возвращала бы вместо значений Avenue, Boulevard и Parade их топографические сокращения, и примените её к данным о подтипах улиц."""

display(melb_df['StreetType'].value_counts())

# melb_df['StreetType'] = melb_df['StreetType'].apply(lambda x: x if x != 'other' in melb_df['StreetType'] else melb_df['StreetType'].drop(['StreetType'], axis = 1))

def rename_street(name):
    if name == 'Avenue':
        return 'Av'
    elif name == 'Boulevard':
        return 'Bvd'
    elif name == 'Parade':
        return 'Pde'
    else:
        return name

street_types = street_types.apply(rename_street)

display(street_types.value_counts())

popular_stypes = street_types.value_counts().nlargest(10).index
print(popular_stypes)
# Index(['St', 'Rd', 'Ct', 'Dr', 'Av', 'Gr', 'Pde', 'Pl', 'Cr', 'Cl'], dtype='objec

melb_df['StreetType'] = street_types.apply(lambda x: x if x in popular_stypes else 'other')

display(melb_df['StreetType'])

print(melb_df['StreetType'].nunique())



In [ ]:
top_49 = melb_df['SellerG'].value_counts().nlargest(49).index

print(top_49)

melb_df['SellerG'] = melb_df['SellerG'].apply(lambda x: x if x in top_49 else 'other')

"""Найдите, во сколько раз минимальная цена объектов недвижимости, проданных компанией 
'Nelson', больше минимальной цены объектов, проданных компаниями, обозначенными как 'other'. Ответ округлите до десятых."""

nelson_min = melb_df[melb_df['SellerG'] == 'Nelson']['Price'].min()

other_min = melb_df[melb_df['SellerG'] == 'other']['Price'].min()

result = round(nelson_min/other_min, 1)

print(result)


## Число уникальных категорий в каждом столбце ##

In [ ]:
# создаём пустой список
unique_list = []
# пробегаемся по именам столбцов в таблице
for col in melb_df.columns:
    # создаём кортеж (имя столбца, число уникальных значений)
    item = (col, melb_df[col].nunique(),melb_df[col].dtypes) 
    # добавляем кортеж в список
    unique_list.append(item) 
# создаём вспомогательную таблицу и сортируем её
unique_counts = pd.DataFrame(
    unique_list,
    columns=['Column_Name', 'Num_Unique', 'Type']
).sort_values(by='Num_Unique',  ignore_index=True)
# выводим её на экран
display(unique_counts)

display(melb_df.info())

"""Сделаем преобразование столбцов к типу данных category:"""

cols_to_exclude = ['Date', 'Rooms', 'Bedroom', 'Bathroom', 'Car'] # список столбцов, которые мы не берём во внимание
max_unique_count = 150 # задаём максимальное число уникальных категорий
for col in melb_df.columns: # цикл по именам столбцов
    if melb_df[col].nunique() < max_unique_count and col not in cols_to_exclude: # проверяем условие
        melb_df[col] = melb_df[col].astype('category') # преобразуем тип столбца
display(melb_df.info())

print(melb_df['Regionname'].cat.categories)

display(melb_df['Regionname'].cat.codes)

melb_df['Type'] = melb_df['Type'].cat.rename_categories({
    'u': 'unit',
    't': 'townhouse',
    'h': 'house'
})
display(melb_df['Type'])

new_houses_types = pd.Series(['unit', 'house', 'flat', 'flat', 'house'])
new_houses_types = new_houses_types.astype(melb_df['Type'].dtype)
display(new_houses_types)

In [ ]:
import pandas as pd

url = 'https://raw.githubusercontent.com/justmarkham/pandas-videos/master/data/ufo.csv'
ufo = pd.read_csv(url)

display(ufo.head())

ufo['Time'] = pd.to_datetime(ufo['Time'])
display(ufo.head())

"""В каком году отмечается наибольшее количество случаев наблюдения НЛО в США?"""
ufo_years = ufo['Time'].dt.year
ufo_years_max = ufo_years.mode()[0]
print(ufo_years_max)

"""Найдите средний интервал времени (в днях) между двумя последовательными случаями наблюдения НЛО в штате Невада (NV)."""

ufo_nv = ufo[ufo['State'] == 'NV']
ufo_days = ufo_nv['Time'].dt.date

# 1. Оставляем только Неваду
ufo_nv = ufo[ufo['State'] == 'NV']

# 2. Достаем даты (без времени)
ufo_days = ufo_nv['Time'].dt.date

# 3. Считаем разницу между соседними строками (получаем интервалы)
"""!!! Очень Важно!!!! """
ufo_diff = ufo_days.diff()

# 4. Берем только количество дней из этих интервалов (.dt.days)
# и считаем среднее (.mean())
result = ufo_diff.dt.days.mean()

# 5. Выводим результат
print(result)

In [ ]:
"""При преобразовании столбцов таблицы о недвижимости к типу category мы оставили без внимания столбец Suburb (пригород). Давайте исправим это.
С помощью метода info() узнайте, сколько памяти занимает таблица melb_df.
Преобразуйте признак Suburb следующим образом: оставьте в столбце только 119 наиболее популярных пригородов, остальные замените на 'other'.
Приведите данные в столбце Suburb к категориальному типу.
В качестве ответа запишите разницу между объёмом занимаемой памяти до преобразования (который мы получили ранее в модуле) и после него в Мб. Ответ округлите до десятых."""

melb_df['Suburb'] = melb_df['Suburb'].astype('object')

melb_df.info()

mem_before = melb_df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Память ДО: {mem_before:.4f} МБ") 


sub_pop_119 = melb_df['Suburb'].value_counts().nlargest(119).index

display(melb_df['Suburb'])

melb_df['Suburb'] = melb_df['Suburb'].apply(lambda x: x if x in sub_pop_119 else 'other')

melb_df['Suburb'] = melb_df['Suburb'].astype('category')

mem_after = melb_df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Память ПОСЛЕ: {mem_after:.4f} МБ")

diff = round(mem_before - mem_after, 1)
print(f"Разница: {diff} МБ")

melb_df.info()


In [ ]:
import pandas as pd

url = 'https://raw.githubusercontent.com/justmarkham/pandas-videos/master/data/ufo.csv'
ufo = pd.read_csv(url)

display(ufo.head())

ufo['Time'] = pd.to_datetime(ufo['Time'])
display(ufo.head())

"""В каком году отмечается наибольшее количество случаев наблюдения НЛО в США?"""
ufo_years = ufo['Time'].dt.year
ufo_years_max = ufo_years.mode()[0]
print(ufo_years_max)

"""Найдите средний интервал времени (в днях) между двумя последовательными случаями наблюдения НЛО в штате Невада (NV)."""

ufo_nv = ufo[ufo['State'] == 'NV']
ufo_days = ufo_nv['Time'].dt.date

# 1. Оставляем только Неваду
ufo_nv = ufo[ufo['State'] == 'NV']

# 2. Достаем даты (без времени)
ufo_days = ufo_nv['Time'].dt.date

# 3. Считаем разницу между соседними строками (получаем интервалы)
"""!!! Очень Важно!!!! """
ufo_diff = ufo_days.diff()

# 4. Берем только количество дней из этих интервалов (.dt.days)
# и считаем среднее (.mean())
result = ufo_diff.dt.days.mean()

# 5. Выводим результат
print(result)

In [ ]:
import pandas as pd

url = 'https://raw.githubusercontent.com/justmarkham/pandas-videos/master/data/ufo.csv'
ufo = pd.read_csv(url)

display(ufo.head())

ufo['Time'] = pd.to_datetime(ufo['Time'])
display(ufo.head())

"""В каком году отмечается наибольшее количество случаев наблюдения НЛО в США?"""
ufo_years = ufo['Time'].dt.year
ufo_years_max = ufo_years.mode()[0]
print(ufo_years_max)

"""Найдите средний интервал времени (в днях) между двумя последовательными случаями наблюдения НЛО в штате Невада (NV)."""

ufo_nv = ufo[ufo['State'] == 'NV']
ufo_days = ufo_nv['Time'].dt.date

# 1. Оставляем только Неваду
ufo_nv = ufo[ufo['State'] == 'NV']

# 2. Достаем даты (без времени)
ufo_days = ufo_nv['Time'].dt.date

# 3. Считаем разницу между соседними строками (получаем интервалы)
"""!!! Очень Важно!!!! """
ufo_diff = ufo_days.diff()

# 4. Берем только количество дней из этих интервалов (.dt.days)
# и считаем среднее (.mean())
result = ufo_diff.dt.days.mean()

# 5. Выводим результат
print(result)

In [ ]:


# 1. Импортируем библиотеки
import numpy as np
import pandas as pd

bike_data.info()

# 2. Загружаем данные (с учетом папки 'data/')
bike_data = pd.read_csv('data/citibike-tripdata.csv')




# bike_data['birth year'].max()
# bike_data['start station name'].value_counts().index[-1]

bike_data['end station name'].value_counts().index[0]

# bike_data['start station name'].nunique()
# #ответ 759
bike_data['start station name'].nunique()
#ответ 765
"""Задание 6.8
В первую очередь удалим лишнюю информацию из данных.
В наших данных присутствуют столбцы, которые дублируют информацию друг о друге: это столбцы с идентификатором и названием стартовой и конечной стоянки. 
Удалите признаки идентификаторов стоянок. Сколько столбцов осталось?"""

# bike_data = bike_data.drop(['start station id', 'end station id'], axis = 1)
# bike_data.shape[1]

"""Замените признак birth year на более понятный признак возраста клиента age. Годом отсчёта возраста выберите 2018 год.
Столбец birth year удалите из таблицы. Сколько поездок совершено клиентами старше 60 лет?"""

# bike_data['age'] = 2018 - bike_data['birth year']
# # display(bike_data['age'])
# bike_data = bike_data.drop(columns = ['birth year'])

# bike_data.info()

# (bike_data['age'] > 60).sum()

"""Задание 6.10
Создайте признак длительности поездки trip duration.
Для этого вычислите интервал времени между временем окончания поездки (stoptime) и её началом (starttime).
Сколько целых минут длилась поездка под индексом 3 в таблице?"""

display(bike_data['stoptime'])

bike_data['stoptime', 'stoptime'] = bike_data.to_datetime(bike_data['stoptime', 'stoptime'])



# bike_data['trip duration'] = (bike_data['stoptime']).astype('int8') - (bike_data['starttime']).astype('in8')

In [ ]:
import numpy as np
import pandas as pd

# from pathlib import Path

# # Получаем текущую рабочую директорию (где запущен блокнот)
# current_dir = Path.cwd() 

# # Собираем путь к файлу (если бы он лежал в папке data)
# file_path = current_dir / 'data' / 'citibike-tripdata.csv'

# # Хорошая практика: проверить, существует ли файл, перед чтением
# if file_path.exists():
#     bike_data = pd.read_csv(file_path)
#     print("Файл успешно загружен!")
# else:
#     print(f"Упс, файла по пути {file_path} нет.")

bike_data = pd.read_csv('data/citibike-tripdata.csv')


# bike_data.head()

"""Задание 6.1
Сколько пропусков в столбце start station id?"""

bike_data.info()

# diff = 300000 - 299831
# print(diff)

# missing_count = bike_data['start station id'].isnull().sum()
# print(missing_count)

"""Задание 6.2
Какой тип данных имеют столбцы starttime и stoptime"""

# bike_data.dtypes

# bike_data[['starttime', 'stoptime']].dtypes

"""Задание 6.3
Найдите идентификатор самой популярной стартовой стоянки. Запишите идентификатор в виде целого числа."""

# bike_data['start station id'].value_counts().index[0]

# round(bike_data['start station id'].mode()[0], 0)

# bike_data['start station id'].value_counts().idxmax()

"""Задание 6.4
Велосипед с каким идентификатором является самым популярным?"""

# bike_data['bikeid'].value_counts().idxmax()

"""Задание 6.5
Какой тип клиентов (столбец usertype) является преобладающим — Subscriber или Customer? 
В качестве ответа запишите долю клиентов преобладающего типа среди общего количества клиентов. Ответ округлите до сотых."""

# bike_data['usertype'].value_counts(normalize=True).round(2)

# bike_data['usertype'].value_counts(normalize=True).round(2).max()

# display(bike_data['usertype'])

"""Задание 6.6
Кто больше занимается велоспортом — мужчины или женщины? В ответ запишите число поездок для той группы, у которой их больше."""


# bike_data['gender'].value_counts().mode()

# bike_data['gender'].value_counts().max()

# bike_data['gender'].value_counts().idxmax()

"""Задание 6.7
Выберите утверждения,которые соответствуют нашим данным:"""

"""Задание 6.10
Создайте признак длительности поездки trip duration.
Для этого вычислите интервал времени между временем окончания поездки (stoptime) и её началом (starttime).
Сколько целых минут длилась поездка под индексом 3 в таблице?"""

display(bike_data['stoptime'])

bike_data[['starttime', 'stoptime']] = bike_data[['starttime', 'stoptime']].apply(pd.to_datetime)


bike_data['trip duration'] = (bike_data['stoptime']) - (bike_data['starttime'])

bike_data['trip duration'] = bike_data['trip duration'].dt.total_seconds()/60

int(bike_data['trip duration'].loc[3])